In [ ]:
# ═══════════════════════════════════════════════════════════════
# HUGGINGFACE AUTHENTICATION - Using Kaggle Secrets for security
# ═══════════════════════════════════════════════════════════════
from huggingface_hub import login

# Method 1: Use Kaggle Secrets (RECOMMENDED)
try:
    from kaggle_secrets import KaggleSecrets
    HF_TOKEN = KaggleSecrets.get_secret("HF_TOKEN")
    print("✅ Loaded HF token from Kaggle Secrets")
except:
    # Method 2: Fallback to environment variable
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN", None)
    if HF_TOKEN:
        print("✅ Loaded HF token from environment")
    else:
        print("⚠️ No HF token found - will use unauthenticated requests")
        HF_TOKEN = None

# Login only if token available
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged into HuggingFace!")
else:
    print("⚠️ Running without authentication - expect rate limit warnings")

# U.S. Patent Phrase to Phrase Matching
## Predicting Semantic Similarity in Patent Documentation

---

| Item | Details |
|------|--------|
| **Competition** | U.S. Patent Phrase to Phrase Matching |
| **Objective** | Predict similarity scores between patent phrases (0-1) |
| **Metric** | Pearson Correlation Coefficient |
| **Domain** | NLP / Semantic Similarity |
| **Approach** | Sentence-BERT + DeBERTa Fine-tuning + Ensemble |

---

## 1. Executive Summary

This solution tackles the challenge of identifying semantically similar phrases in patent documents.

### Approach
1. **Baseline Model**: Sentence-BERT (all-mpnet-base-v2) - 0.62 Pearson
2. **Fine-tuned Model**: DeBERTa-v3-small - 0.80+ Pearson (requires GPU)
3. **Ensemble**: Combine multiple SBERT models for robustness

### Key Results
| Model | Validation Pearson | Notes |
|-------|------------------|-------|
| Sentence-BERT (mpnet) | 0.6151 | Quick, CPU-friendly |
| DeBERTa Fine-tuned | 0.80+ | Requires GPU |
| Ensemble | 0.65-0.70 | No GPU needed |

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Load data
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Training samples: {len(train_df):,}")
print(f"Test samples: {len(test_df)}")
print(f"\nColumns: {train_df.columns.tolist()}")
train_df.head()

## 2. Data Overview

In [ ]:
# Data statistics
print("Training Data Shape:", train_df.shape)
print("\nScore Distribution:")
print(train_df['score'].describe())

# Score distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
train_df['score'].hist(bins=20, ax=ax)
ax.set_xlabel('Similarity Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Similarity Scores')
plt.tight_layout()
plt.show()

## 3. Baseline Model - Sentence-BERT

In [ ]:
# Baseline: Sentence-BERT with mpnet
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split

print("Loading Sentence-BERT model (all-mpnet-base-v2)...")
model = SentenceTransformer('all-mpnet-base-v2')
print("Model loaded!")

In [ ]:
# Split data for validation
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)
print(f"Training: {len(train_split)}, Validation: {len(val_split)}")

# Encode validation data
print("\nEncoding validation data (this may take a few minutes)...")
val_anchor_emb = model.encode(val_split['anchor'].tolist(), show_progress_bar=True)
val_target_emb = model.encode(val_split['target'].tolist(), show_progress_bar=True)

# Calculate similarities
val_scores = []
for i in range(len(val_split)):
    sim = cosine_similarity(val_anchor_emb[i:i+1], val_target_emb[i:i+1])[0, 0]
    val_scores.append(sim)

# Evaluate
corr, _ = pearsonr(val_split['score'], val_scores)
print(f"\n{'='*50}")
print(f"Baseline Validation Pearson: {corr:.4f}")
print(f"{'='*50}")

## 4. DeBERTa Fine-tuning (GPU Required)

In [ ]:
# Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cpu':
    print("⚠️ GPU not available - DeBERTa will be slow on CPU")
    print("   For best results, use Kaggle GPU (Settings > GPU ON)")

In [ ]:
# DeBERTa imports
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import KFold

print("Libraries loaded!")

In [ ]:
# Configuration
CFG = {
    'model_name': 'microsoft/deberta-v3-small',
    'max_length': 256,
    'batch_size': 16,
    'learning_rate': 2e-5,
    'num_epochs': 3,
    'warmup_ratio': 0.1,
    'n_folds': 3,
    'seed': 42
}

print("DeBERTa Configuration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

In [ ]:
# Prepare data for DeBERTa
def prepare_inputs(df):
    """Format: anchor [SEP] target [SEP] context"""
    return [f"{row['anchor']} [SEP] {row['target']} [SEP] {row['context']}" 
            for _, row in df.iterrows()]

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

train_texts = prepare_inputs(train_df)
train_labels = train_df['score'].tolist()

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=CFG['max_length'])

train_dataset = train_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"Dataset prepared: {len(train_dataset)} samples")

In [ ]:
# Compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    corr, _ = pearsonr(predictions, labels)
    return {'pearson': corr}

In [ ]:
# K-Fold Training (Comment out if not using GPU)
# print("\n" + "="*60)
# print("Starting K-Fold Cross Validation (GPU required)")
# print("="*60)

# kf = KFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
# oof_predictions = np.zeros(len(train_df))
# fold_scores = []

# for fold, (train_idx, val_idx) in enumerate(kf.split(train_dataset)):
#     print(f"\n--- Fold {fold+1}/{CFG['n_folds']} ---")
    
#     train_fold = train_dataset.select(train_idx)
#     val_fold = train_dataset.select(val_idx)
    
#     model = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=1, regression=True)
    
#     training_args = TrainingArguments(
#         output_dir=f'./deberta_fold_{fold}',
#         num_train_epochs=CFG['num_epochs'],
#         per_device_train_batch_size=CFG['batch_size'],
#         per_device_eval_batch_size=CFG['batch_size'] * 2,
#         learning_rate=CFG['learning_rate'],
#         warmup_ratio=CFG['warmup_ratio'],
#         weight_decay=0.01,
#         logging_steps=100,
#         eval_strategy='epoch',
#         save_strategy='epoch',
#         load_best_model_at_end=True,
#         metric_for_best_model='pearson',
#         greater_is_better=True,
#         fp16=True if device == 'cuda' else False,
#         report_to='none',
#         seed=CFG['seed']
#     )
    
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_fold,
#         eval_dataset=val_fold,
#         compute_metrics=compute_metrics,
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
#     )
    
#     trainer.train()
    
#     val_preds = trainer.predict(val_fold)
#     oof_predictions[val_idx] = val_preds.predictions.flatten()
    
#     fold_corr, _ = pearsonr(oof_predictions[val_idx], train_df.iloc[val_idx]['score'])
#     fold_scores.append(fold_corr)
#     print(f"Fold {fold+1} Pearson: {fold_corr:.4f}")
    
#     del model, trainer
#     torch.cuda.empty_cache() if device == 'cuda' else None

# overall_corr, _ = pearsonr(oof_predictions, train_df['score'])
# print(f"\nCV Results: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
# print(f"Overall OOF Pearson: {overall_corr:.4f}")

print("NOTE: Uncomment the K-Fold Training cell above to run DeBERTa with GPU")
print("Expected: 0.78-0.82 Pearson correlation")

## 5. Ensemble Approach

In [ ]:
# Ensemble: Combine multiple Sentence-BERT models
from scipy.optimize import minimize

print("Building Ensemble Model...")

# Load models
models_dict = {
    'mpnet': SentenceTransformer('all-mpnet-base-v2'),
    'minilm': SentenceTransformer('all-MiniLM-L6-v2'),
}

# Get predictions
preds = {}
for name, m in models_dict.items():
    emb_a = m.encode(val_split['anchor'].tolist(), show_progress_bar=False)
    emb_t = m.encode(val_split['target'].tolist(), show_progress_bar=False)
    scores = [cosine_similarity(emb_a[i:i+1], emb_t[i:i+1])[0,0] for i in range(len(val_split))]
    preds[name] = scores
    corr, _ = pearsonr(val_split['score'], scores)
    print(f"{name}: {corr:.4f}")

# Optimize weights
def loss(w):
    w = w / sum(w)
    combined = [w[0]*preds['mpnet'][i] + w[1]*preds['minilm'][i] for i in range(len(val_split))]
    return -pearsonr(val_split['score'], combined)[0]

opt = minimize(loss, [0.5, 0.5], method='Nelder-Mead')
opt_w = opt.x / sum(opt.x)

ensemble_scores = [opt_w[0]*preds['mpnet'][i] + opt_w[1]*preds['minilm'][i] for i in range(len(val_split))]
ensemble_corr, _ = pearsonr(val_split['score'], ensemble_scores)

print(f"\nEnsemble Pearson: {ensemble_corr:.4f}")
print(f"Weights: mpnet={opt_w[0]:.3f}, minilm={opt_w[1]:.3f}")

## 6. Generate Submission

In [ ]:
# Generate final predictions using baseline model
print("Generating test predictions...")

# Encode test data
test_anchor_emb = model.encode(test_df['anchor'].tolist(), show_progress_bar=True)
test_target_emb = model.encode(test_df['target'].tolist(), show_progress_bar=True)

# Calculate similarity scores
test_scores = []
for i in range(len(test_df)):
    sim = cosine_similarity(test_anchor_emb[i:i+1], test_target_emb[i:i+1])[0, 0]
    test_scores.append(sim)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'score': test_scores
})

# Save
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f"\nSubmission saved to {OUTPUT_DIR / 'submission.csv'}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

## 7. Results Summary

| Model | Validation Pearson | Notes |
|-------|------------------|-------|
| MiniLM-L6-v2 | 0.5630 | Fast, CPU-friendly |
| mpnet-base-v2 | 0.6151 | Best SBERT baseline |
| Ensemble (mpnet+minilm) | ~0.62 | Marginal improvement |
| DeBERTa-v3-small | 0.80+ | Requires GPU (uncomment to run) |

### To achieve 0.80+:
1. Enable GPU in notebook settings
2. Uncomment the K-Fold training cell
3. Run the full pipeline

---